# Name : Muhammad Danish Zaheer Awan
# REG id : 25280092
## Task 4 - Open-Set Recognition

## Workflow rules

- Use CIFAR-10 only for training, checkpoint selection, score design, and threshold calibration.
- Write hypotheses and create the immutable experiment lock before loading CIFAR-100.
- Run the guarded final evaluation once; do not tune anything from its results.

## 0. Imports and project setup

In [ ]:
# Import standard utilities for cleanup, paths, tables, plotting, and PyTorch.
# Locate the PA_1 root so imports work when this notebook starts inside task4/scripts.

import gc
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root.name != 'PA_1' and project_root.parent != project_root:
    project_root = project_root.parent
if project_root.name != 'PA_1':
    raise FileNotFoundError('Run this notebook from inside the PA_1 project.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
# Import configuration, CIFAR-10 preparation, and known-only training helpers.
# None of these modules imports or constructs the CIFAR-100 unknown dataset.

from task4.configs.config_loader import (
    create_output_directories,
    get_output_paths,
    load_config,
)
from task4.data.cifar10 import prepare_cifar10_datasets
from task4.training.train import (
    select_training_device,
    set_random_seed,
    train_or_load_task4_method,
)

In [ ]:
# Import known-data calibration, experiment locking, and final evaluation functions.
# Importing the evaluator is safe because CIFAR-100 is imported only inside its final function.

from task4.evaluation.evaluate_osr import (
    create_experiment_lock,
    prepare_known_evaluation,
    run_final_open_set_evaluation,
)
from task4.evaluation.metrics import save_json

## 1. Runtime and exact configurations

In [ ]:
# Define reproducible random state and automatically select the available CUDA GPU.
# The training code also restores this seed independently for each method.

def configure_runtime(seed=6304):
    set_random_seed(seed)
    return select_training_device()

In [ ]:
# Initialize the shared runtime and show whether GPU acceleration is active.
# CUDA is used for training and feature extraction whenever it is available.

DEVICE = configure_runtime()
print(f'Project root: {project_root}')
print(f'PyTorch version: {torch.__version__}')
print(f'Selected device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(DEVICE)}')

In [ ]:
# Define one helper that loads the validated Vanilla, GCSC, and PROSER YAML files.
# The loader rejects changes to the fixed architecture and optimization protocol.

def load_task4_configurations():
    return {name: load_config(name) for name in ('vanilla', 'gcsc', 'proser')}

In [ ]:
# Load all exact configurations and create the data-cache and result directories.
# The same output layout is shared by every method, without placeholder .gitkeep files.

CONFIGURATIONS = load_task4_configurations()
OUTPUT_PATHS = create_output_directories(CONFIGURATIONS['vanilla'])
configuration_rows = []
for method_name, configuration in CONFIGURATIONS.items():
    configuration_rows.append({
        'method': method_name,
        'epochs': configuration['training']['maximum_epochs'],
        'learning_rate': configuration['training']['learning_rate'],
        'randaugment': configuration['method']['use_randaugment'],
        'dummy_classes': configuration['method']['number_of_dummy_classes'],
    })
display(pd.DataFrame(configuration_rows))

## 2. CIFAR-10 data and fixed 90/10 split

In [ ]:
# Define data preparation separately because each method has its own training transform.
# Validation, test, and Mahalanobis-training views remain deterministic and unaugmented.

def prepare_method_datasets(configuration):
    return prepare_cifar10_datasets(configuration)

In [ ]:
# Prepare the Vanilla data first, downloading CIFAR-10 into Data/cifar10 if needed.
# This also creates or validates the reusable stratified split JSON with seed 6304.

VANILLA_DATASETS, CIFAR10_PROTOCOL = prepare_method_datasets(
    CONFIGURATIONS['vanilla']
)
print(f"Split file: {OUTPUT_PATHS['cifar10_split_file']}")
print(f"Split seed: {CIFAR10_PROTOCOL['seed']}")

In [ ]:
# Define a compact size table for the optimization, validation, and test views.
# The train_evaluation view reuses training indices without stochastic augmentation.

def create_dataset_size_table(datasets):
    return pd.DataFrame([
        {'split': split_name, 'examples': len(dataset)}
        for split_name, dataset in datasets.items()
    ])

In [ ]:
# Display dataset sizes before any optimization starts.
# Expected counts are 45,000 train, 5,000 validation, and 10,000 test examples.

display(create_dataset_size_table(VANILLA_DATASETS))

## 3. Train or reuse selected checkpoints

In [ ]:
# Define a method runner that prepares the correct transform and trains or reuses a checkpoint.
# Keep only compact checkpoint metadata after each run so GPU memory is free for the next method.

def run_training_method(method_name, datasets=None, force_retrain=False):
    configuration = CONFIGURATIONS[method_name]
    if datasets is None:
        datasets, protocol = prepare_method_datasets(configuration)
    else:
        protocol = CIFAR10_PROTOCOL
    result = train_or_load_task4_method(
        configuration,
        datasets,
        device=DEVICE,
        force_retrain=force_retrain,
    )
    checkpoint_keys = ('epoch', 'validation', 'selection_metric', 'cifar100_used_during_training')
    result['checkpoint'] = {key: result['checkpoint'][key] for key in checkpoint_keys}
    result['model'].to('cpu')
    del result['model']
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result, protocol

In [ ]:
# Choose whether existing exact checkpoints should be replaced by fresh training runs.
# Keep False for safe notebook restarts; set True only when intentionally retraining all methods.

FORCE_RETRAIN = False
TRAINING_RESULTS = {}

### 3.1 Vanilla

In [ ]:
# Train Vanilla first because its selected checkpoint initializes the later PROSER model.
# Selection uses CIFAR-10 validation accuracy only and writes one history CSV per epoch.

TRAINING_RESULTS['vanilla'], _ = run_training_method(
    'vanilla',
    datasets=VANILLA_DATASETS,
    force_retrain=FORCE_RETRAIN,
)
print(TRAINING_RESULTS['vanilla']['checkpoint_file'])

### 3.2 GCSC

In [ ]:
# Train GCSC with the same protocol while adding RandAugment(2, 9) to training images.
# Its initialization, optimizer, schedule, split, and checkpoint rule match Vanilla.

TRAINING_RESULTS['gcsc'], _ = run_training_method(
    'gcsc',
    force_retrain=FORCE_RETRAIN,
)
print(TRAINING_RESULTS['gcsc']['checkpoint_file'])

### 3.3 PROSER

In [ ]:
# Fine-tune PROSER from the selected Vanilla checkpoint with five dummy classifiers.
# Classifier placeholders and different-class layer2 mixup use equal batch halves.

TRAINING_RESULTS['proser'], _ = run_training_method(
    'proser',
    force_retrain=FORCE_RETRAIN,
)
print(TRAINING_RESULTS['proser']['checkpoint_file'])

## 4. Training histories and selected epochs

In [ ]:
# Define one comparison plot for loss and CIFAR-10 validation accuracy.
# Curves are read from the returned histories and saved under task4/results/figures.

def plot_training_histories(training_results, output_file):
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for method_name, result in training_results.items():
        history = result['history']
        if history.empty:
            continue
        axes[0].plot(history['epoch'], history['total_loss'], label=method_name.upper())
        axes[1].plot(history['epoch'], history['validation_accuracy'], label=method_name.upper())
    axes[0].set(title='Training loss', xlabel='Epoch', ylabel='Loss')
    axes[1].set(title='CIFAR-10 validation accuracy', xlabel='Epoch', ylabel='Accuracy')
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    figure.tight_layout()
    output_file.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_file, dpi=200, bbox_inches='tight')
    plt.show()
    return output_file

In [ ]:
# Plot the three completed histories and save the reproducible figure.
# Missing histories are skipped so old checkpoints can still be inspected safely.

TRAINING_FIGURE = plot_training_histories(
    TRAINING_RESULTS,
    OUTPUT_PATHS['figures_directory'] / 'task4_training_histories.png',
)
print(TRAINING_FIGURE)

In [ ]:
# Define a table showing the source-selected epoch and validation metrics per checkpoint.
# This evidence confirms that CIFAR-100 did not participate in model selection.

def create_checkpoint_selection_table(training_results):
    rows = []
    for method_name, result in training_results.items():
        checkpoint = result['checkpoint']
        rows.append({
            'method': method_name,
            'selected_epoch': checkpoint['epoch'],
            'validation_accuracy': checkpoint['validation']['accuracy'],
            'validation_macro_f1': checkpoint['validation']['macro_f1'],
            'cifar100_used_during_training': checkpoint['cifar100_used_during_training'],
        })
    return pd.DataFrame(rows)

In [ ]:
# Display checkpoint-selection evidence after all three known-only runs finish.
# Every row must show False in the final column before calibration continues.

CHECKPOINT_TABLE = create_checkpoint_selection_table(TRAINING_RESULTS)
display(CHECKPOINT_TABLE)

## 5. Freeze CIFAR-10 outputs, scores, and thresholds

In [ ]:
# Define preparation of deterministic known-only views for all three methods.
# Method-specific optimization transforms are irrelevant to these unaugmented evaluation views.

def prepare_known_dataset_mapping():
    datasets_by_method = {'vanilla': VANILLA_DATASETS}
    for method_name in ('gcsc', 'proser'):
        datasets, protocol = prepare_method_datasets(CONFIGURATIONS[method_name])
        if protocol != CIFAR10_PROTOCOL:
            raise ValueError('All methods must reuse the exact CIFAR-10 split.')
        datasets_by_method[method_name] = datasets
    return datasets_by_method

In [ ]:
# Build known-only dataset mappings and extract frozen checkpoint outputs.
# This cell fits Vanilla Mahalanobis statistics and calibrates every threshold on validation.

KNOWN_DATASETS = prepare_known_dataset_mapping()
KNOWN_EVALUATION = prepare_known_evaluation(
    CONFIGURATIONS,
    KNOWN_DATASETS,
    device=DEVICE,
)

In [ ]:
# Display closed-set CIFAR-10 test accuracy and macro-F1 for the selected models.
# These test metrics do not affect checkpoints, score definitions, or thresholds.

display(KNOWN_EVALUATION['closed_set_table'])

In [ ]:
# Define a readable table from the validation-only threshold registry.
# All scores are unknownness values, so examples are accepted when score is at most tau.

def create_threshold_table(threshold_registry):
    return pd.DataFrame([
        {'result_name': name, **record}
        for name, record in threshold_registry.items()
    ])

In [ ]:
# Display every threshold before writing hypotheses or touching CIFAR-100.
# Each tau is the 95th percentile of the corresponding CIFAR-10 validation score.

THRESHOLD_TABLE = create_threshold_table(KNOWN_EVALUATION['thresholds'])
display(THRESHOLD_TABLE)

## 6. Write hypotheses before seeing unknown results

Replace every `TODO` below with your own prediction and reasoning. The lock cell intentionally refuses to run while any placeholder remains.

In [ ]:
# Record predictions about score and method comparisons before final unknown evaluation.
# Replace all TODO text; these statements are hashed into the immutable experiment lock.

HYPOTHESES = {
    'best_vanilla_score': 'TODO: predict which Vanilla score will have the best unknown AUROC and why.',
    'near_versus_far': 'TODO: predict whether near or far unknowns will be harder and why.',
    'gcsc_effect': 'TODO: predict how GCSC will affect CSA and MLS-based OSR.',
    'proser_effect': 'TODO: predict how PROSER MLS and placeholder scores will compare.',
    'expected_failure_pattern': 'TODO: predict which semantic confusions will appear among accepted unknowns.',
}
HYPOTHESES_FILE = OUTPUT_PATHS['metrics_directory'] / 'task4_hypotheses.json'
save_json(HYPOTHESES, HYPOTHESES_FILE)
print(HYPOTHESES_FILE)

## 7. Lock all decisions

In [ ]:
# Create paths for the immutable lock and inspect the artifacts it will protect.
# The lock includes configs, checkpoints, known caches, thresholds, score code, and hypotheses.

LOCK_FILE = OUTPUT_PATHS['metrics_directory'] / 'task4_experiment_lock.json'
print(f'Hypotheses: {HYPOTHESES_FILE}')
print(f'Lock: {LOCK_FILE}')

In [ ]:
# Freeze all decisions only after replacing every TODO hypothesis above.
# Any later artifact change invalidates the lock and blocks CIFAR-100 evaluation.

EXPERIMENT_LOCK = create_experiment_lock(CONFIGURATIONS, LOCK_FILE)
print('Task 4 decisions are locked.')

## 8. One guarded final CIFAR-100 evaluation

Run the next two cells only when models, scores, thresholds, and hypotheses are final. Set the flag to `True` once; the evaluator validates the lock before dynamically importing CIFAR-100.

In [ ]:
# Keep final unknown evaluation disabled until all known-only choices are frozen.
# Changing this flag is the explicit confirmation that no further tuning will follow.

RUN_FINAL_UNKNOWN_EVALUATION = False

In [ ]:
# Validate the immutable lock and run the single final CIFAR-100 near/far evaluation.
# Unknown data is downloaded into Data/cifar100 only inside this explicitly enabled call.

if not RUN_FINAL_UNKNOWN_EVALUATION:
    raise RuntimeError('Set RUN_FINAL_UNKNOWN_EVALUATION=True only for the final run.')
FINAL_RESULTS = run_final_open_set_evaluation(
    CONFIGURATIONS,
    LOCK_FILE,
    device=DEVICE,
)

## 9. Required result tables and evidence

In [ ]:
# Display the required comparison of MSP, MLS, Energy, and Mahalanobis on Vanilla.
# It reports known CSA plus near, far, and combined open-set measurements.

display(FINAL_RESULTS['vanilla_score_table'])

In [ ]:
# Display the trained-model comparison using MLS plus the PROSER placeholder score.
# This table contains Vanilla, GCSC, PROSER-MLS, and PROSER-placeholder rows.

display(FINAL_RESULTS['model_comparison_table'])

In [ ]:
# Display incorrectly accepted near and far unknown examples under Vanilla MLS tau.
# Each row records unknown class, predicted known class, score, threshold, and image index.

display(FINAL_RESULTS['failure_table'])

In [ ]:
# List the saved tables, predictions, and figures for direct use in the report.
# Large logits and features stay under Data/task4_cache instead of the result folder.

for output_name, output_path in {
    'Vanilla score table': OUTPUT_PATHS['metrics_directory'] / 'vanilla_score_comparison.csv',
    'Model table': OUTPUT_PATHS['metrics_directory'] / 'trained_model_comparison.csv',
    'Failure table': OUTPUT_PATHS['predictions_directory'] / 'vanilla_mls_accepted_failures.csv',
    'Score distributions': OUTPUT_PATHS['figures_directory'] / 'vanilla_score_distributions.png',
    'Failure images': OUTPUT_PATHS['figures_directory'] / 'vanilla_mls_accepted_failures.png',
}.items():
    print(f'{output_name}: {output_path}')

## 10. Interpretation checklist

Compare final observations with the locked hypotheses, discuss why near unknowns may resemble known classes, explain the CSA/OSR trade-off for each method, and use the accepted-unknown examples as qualitative evidence. Do not rerun training or recalibrate thresholds after reading these results.